## 🎯 Learning Objectives
* Understand the core principles and benefits of multi-agent networks in AI.
* Differentiate between supervisor and swarm topologies for agent orchestration.
* Implement a basic supervisor-based multi-agent system using LangGraph.
* Implement a basic swarm-based multi-agent system using LangGraph.
* Analyze the performance trade-offs and typical use cases for each topology.


## Multi-agent Networks: Supervisor and Swarm Topologies

As AI systems tackle increasingly complex, open-ended problems, a single monolithic agent often falls short. The solution lies in **multi-agent networks**, where specialized AI agents collaborate, communicate, and coordinate to achieve a common goal. These networks mimic human organizations, distributing cognitive load and leveraging diverse expertise.

LangGraph, a powerful library built on LangChain, provides the foundational tools to design and implement these sophisticated agentic workflows. It allows us to define states, nodes (agents or tools), and edges (transitions), creating directed acyclic graphs (DAGs) or even cyclic graphs for iterative processes.

### Why Multi-Agent Networks?

1.  **Modularity**: Break down complex problems into smaller, manageable sub-problems, each handled by a specialized agent.
2.  **Robustness**: Failure of one agent doesn't necessarily cripple the entire system if other agents can compensate or re-route tasks.
3.  **Scalability**: Easily add or remove agents to adapt to changing task requirements or computational resources.
4.  **Specialization**: Agents can be fine-tuned for specific tasks (e.g., research, coding, summarization, critique), leading to higher quality outputs.
5.  **Emergent Behavior**: Complex, intelligent behaviors can emerge from the interactions of simpler agents.

### Key Topologies:

#### 1. Supervisor Topology (The Project Manager)

Imagine a project manager overseeing a team of specialists. The **supervisor agent** acts as a central orchestrator. It receives the initial task, breaks it down, and delegates sub-tasks to specialized worker agents. After a worker agent completes its task, the supervisor evaluates the output, decides the next step, and potentially assigns the next task to another agent or requests revisions. This creates a clear, hierarchical flow of control.

**Analogy**: A software development team where a lead developer assigns tasks to front-end, back-end, and QA engineers, reviews their work, and guides the project through different phases.

**Characteristics**:
*   Centralized control.
*   Clear task delegation.
*   Sequential or conditionally sequential execution.
*   Good for tasks requiring strict oversight and quality control.

#### 2. Swarm Topology (The Collaborative Team)

Now, consider a group of experts brainstorming and working in parallel on different aspects of a problem, then collectively synthesizing their findings. In a **swarm topology**, agents operate with more autonomy, often in parallel or in a less strictly hierarchical manner. They might share a common goal and contribute to a shared state, or work on independent sub-problems whose results are later aggregated or synthesized by another agent.

**Analogy**: A research group where multiple scientists investigate different facets of a complex problem simultaneously, then come together to combine their findings into a comprehensive report.

**Characteristics**:
*   Decentralized or semi-decentralized control.
*   Parallel execution of tasks.
*   Agents might communicate directly or through a shared state.
*   Often involves an aggregation or synthesis step.
*   Excellent for tasks requiring diverse perspectives, exploration, or high throughput.

Both topologies have their strengths and weaknesses, and the choice depends on the specific problem, desired level of control, and performance requirements. LangGraph allows us to build both, and even hybrid models, by defining nodes and conditional edges that dictate the flow of information and control.


In [ ]:
import operator
from typing import Annotated, List, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.pydantic_v1 import Field
from langgraph.graph import StateGraph, END

# --- 1. Define the Graph State ---
# This defines the object that is passed between nodes in the graph.
# It's a TypedDict for clarity and type safety.
class AgentState(TypedDict):
    task: str
    research_results: Annotated[List[str], Field(default_factory=list)]
    draft_content: str
    final_content: str
    next_agent: str # Used by supervisor to decide next step

# --- 2. Define Agent Nodes (Simulated LLM calls) ---
# In a real application, these would be actual LLM calls or tool invocations.

def research_agent(state: AgentState) -> AgentState:
    print("---RESEARCH AGENT---")
    task = state["task"]
    # Simulate research by finding keywords related to the task
    keywords = f"keywords for '{task}'"
    research_output = f"Found key concepts: {keywords}. Relevant data points: [Data A, Data B]."
    return {"research_results": state["research_results"] + [research_output], "next_agent": "writer"}

def writer_agent(state: AgentState) -> AgentState:
    print("---WRITER AGENT---")
    task = state["task"]
    research_results = state["research_results"]
    # Simulate writing based on research
    draft = f"Initial draft for '{task}' based on research: {research_results[0] if research_results else 'No research provided.'}"
    return {"draft_content": draft, "next_agent": "editor"}

def editor_agent(state: AgentState) -> AgentState:
    print("---EDITOR AGENT---")
    draft = state["draft_content"]
    # Simulate editing and refining the draft
    final_version = f"Edited and refined version of: '{draft}'. Added clarity and flow."
    return {"final_content": final_version, "next_agent": "FINISH"}

def supervisor_agent(state: AgentState) -> AgentState:
    print("---SUPERVISOR AGENT---")
    # In a real scenario, this would involve an LLM call to decide the next step
    # based on the current state and task progress.
    # For this example, we'll use the 'next_agent' field set by previous agents.
    current_next_agent = state.get("next_agent", "researcher")
    print(f"Supervisor decides next: {current_next_agent}")
    return {"next_agent": current_next_agent}

# --- Swarm Agents (Parallel Processing Example) ---
def brainstorm_agent_A(state: AgentState) -> AgentState:
    print("---BRAINSTORM AGENT A---")
    task = state["task"]
    idea_a = f"Idea A for '{task}': Focus on historical context."
    return {"research_results": state["research_results"] + [idea_a]}

def brainstorm_agent_B(state: AgentState) -> AgentState:
    print("---BRAINSTORM AGENT B---")
    task = state["task"]
    idea_b = f"Idea B for '{task}': Focus on future implications."
    return {"research_results": state["research_results"] + [idea_b]}

def aggregator_agent(state: AgentState) -> AgentState:
    print("---AGGREGATOR AGENT---")
    all_ideas = state["research_results"]
    combined_output = f"Combined ideas: {'; '.join(all_ideas)}. Synthesized into a cohesive plan."
    return {"final_content": combined_output}


# --- 3. Build the Supervisor Graph ---
print("\n--- Building Supervisor Graph ---")
workflow_supervisor = StateGraph(AgentState)

workflow_supervisor.add_node("researcher", research_agent)
workflow_supervisor.add_node("writer", writer_agent)
workflow_supervisor.add_node("editor", editor_agent)
workflow_supervisor.add_node("supervisor", supervisor_agent)

# Define the entry point
workflow_supervisor.set_entry_point("supervisor")

# Define conditional edges based on supervisor's decision
workflow_supervisor.add_conditional_edges(
    "supervisor",
    lambda state: state["next_agent"],
    {
        "researcher": "researcher",
        "writer": "writer",
        "editor": "editor",
        "FINISH": END
    },
)

# Define direct edges for worker agents
workflow_supervisor.add_edge("researcher", "supervisor")
workflow_supervisor.add_edge("writer", "supervisor")
workflow_supervisor.add_edge("editor", "supervisor")

# Compile the graph
app_supervisor = workflow_supervisor.compile()

# --- Run the Supervisor Graph ---
print("\n--- Running Supervisor Graph Example ---")
initial_state_supervisor = {"task": "Explain quantum computing to a high school student", "research_results": [], "draft_content": "", "final_content": "", "next_agent": "researcher"}

for s in app_supervisor.stream(initial_state_supervisor):
    print(s)
    print("----")

print("\nFinal Supervisor Output:")
print(app_supervisor.invoke(initial_state_supervisor)["final_content"])


# --- 4. Build the Swarm Graph (Simplified Parallel Example) ---
print("\n--- Building Swarm Graph ---")
workflow_swarm = StateGraph(AgentState)

workflow_swarm.add_node("brainstorm_A", brainstorm_agent_A)
workflow_swarm.add_node("brainstorm_B", brainstorm_agent_B)
workflow_swarm.add_node("aggregator", aggregator_agent)

# Define the entry point
workflow_swarm.set_entry_point("brainstorm_A") # Start with one, then branch

# Branch to run brainstorm_B in parallel (conceptually, or sequentially if no parallel executor)
# For true parallelism, you'd use a custom executor or a more advanced LangGraph feature.
# Here, we'll simulate by having A lead to B, then both feed to aggregator.
# A more direct parallel would involve a 'join' node or custom state merging.

# For simplicity, let's make them run sequentially but contribute to the same state
# before aggregation. A true parallel execution would require a custom executor
# or a more complex state management with LangGraph's `channels` or `fork/join` patterns.
# For this conceptual example, we'll chain them to demonstrate shared state contribution.
workflow_swarm.add_edge("brainstorm_A", "brainstorm_B")
workflow_swarm.add_edge("brainstorm_B", "aggregator")
workflow_swarm.add_edge("aggregator", END)

# Compile the graph
app_swarm = workflow_swarm.compile()

# --- Run the Swarm Graph ---
print("\n--- Running Swarm Graph Example ---")
initial_state_swarm = {"task": "Develop a new marketing strategy", "research_results": [], "draft_content": "", "final_content": "", "next_agent": ""}

for s in app_swarm.stream(initial_state_swarm):
    print(s)
    print("----")

print("\nFinal Swarm Output:")
print(app_swarm.invoke(initial_state_swarm)["final_content"])


### Interpreting the Code Output and Use Cases

#### Supervisor Topology Output

In the supervisor example, you'll observe a clear, sequential flow dictated by the `supervisor_agent`. The `supervisor_agent` acts as a router, deciding which specialized agent (`researcher`, `writer`, `editor`) should execute next based on the `next_agent` field in the shared state. Each worker agent performs its task and then returns control to the supervisor, which then makes the next routing decision. This demonstrates a controlled, step-by-step progression through a workflow.

*   **Performance Trade-offs**: This topology introduces latency due to the sequential nature of tasks and the overhead of the supervisor's decision-making. However, it offers high control, making it easier to debug and ensure quality at each step. If an agent fails, the supervisor can potentially retry or re-route.
*   **Typical Use Cases**:
    *   **Content Generation with Review**: Research -> Draft -> Edit -> Publish, with a supervisor ensuring each stage meets quality standards.
    *   **Complex Problem Solving**: Breaking down a problem into sub-problems, assigning to specialized solvers, and integrating results.
    *   **Automated Customer Support**: A supervisor agent routes queries to different specialized agents (e.g., billing, technical support, product info) based on query intent.
    *   **Code Generation and Refinement**: Plan -> Generate Code -> Test -> Debug -> Refine, with a supervisor managing the iteration.

#### Swarm Topology Output

For the swarm example, we've simulated a simplified parallel contribution model. `brainstorm_agent_A` and `brainstorm_agent_B` both contribute to the `research_results` list in the shared state. Although executed sequentially in this basic LangGraph setup (without a custom parallel executor), conceptually they are working on different aspects of the problem. Finally, the `aggregator_agent` combines their independent contributions into a single `final_content`.

*   **Performance Trade-offs**: A true parallel swarm can significantly reduce overall task completion time by distributing work. However, managing shared state and resolving conflicts or redundancies between agents can be complex. The quality of the final output heavily relies on the effectiveness of the aggregation or synthesis step. Debugging can be harder due to concurrent operations.
*   **Typical Use Cases**:
    *   **Creative Brainstorming**: Multiple agents generate diverse ideas or solutions to a problem, which are then curated or combined.
    *   **Data Analysis from Multiple Sources**: Agents extract insights from different datasets concurrently, and an aggregator synthesizes a comprehensive report.
    *   **Market Research**: Agents analyze different market segments or competitor strategies in parallel, with findings combined for a holistic view.
    *   **Code Review/Testing**: Multiple agents review different parts of a codebase or run different test suites concurrently.

### Choosing the Right Topology

*   **Supervisor**: Opt for this when you need strict control, sequential processing, clear decision points, and robust error handling. It's ideal for tasks where quality assurance and step-by-step validation are critical.
*   **Swarm**: Choose this for tasks that benefit from diverse perspectives, parallel processing, and when the problem can be naturally decomposed into independent or semi-independent sub-problems. It's excellent for exploration, creativity, and speed, provided you have a robust aggregation mechanism.

In practice, many advanced agent systems combine elements of both, creating **hybrid topologies** where a high-level supervisor might manage several sub-swarms, or a swarm might have a mini-supervisor for coordination within its group. LangGraph's flexibility allows for the construction of such intricate architectures.


### Resources

*   **LangGraph Documentation**: The official documentation is the best place to dive deeper into state management, conditional edges, and advanced graph patterns. [https://langchain-ai.github.io/langgraph/](https://langchain-ai.github.io/langgraph/)
*   **LangChain Expression Language (LCEL)**: Understand how to compose chains and agents, which forms the building blocks for LangGraph nodes. [https://python.langchain.com/docs/expression_language/](https://python.langchain.com/docs/expression_language/)
*   **Multi-Agent Systems Research**: Explore academic papers and articles on multi-agent systems for theoretical foundations and advanced concepts. A good starting point is often papers discussing agent communication languages (ACLs) or distributed AI systems.
*   **Agentic AI Frameworks**: Keep an eye on evolving frameworks and tools in the agentic AI space, as this field is rapidly advancing. Platforms like Google AI Studio and Hugging Face often feature new agent-related capabilities and models.
